# P001 0-1 背包问题 · PySCIPOpt 建模与评测演示

本 Notebook 运行在 JupyterLite (WebAssembly + PySCIPOpt) 环境中。
在浏览器内核中可以直接 `from pyscipopt import Model` 求解混合整数线性规划 (MIP)。

In [ ]:
from pyscipopt import Model, quicksum
import numpy as np
print("PySCIPOpt 与 NumPy 在浏览器 WebAssembly 环境中加载成功！")

## 1. 契约定义：solve(instance) -> dict

输入数据规范：
- `n`: 物品数量
- `weights`: 物品重量列表 `[w0, w1, ...]`
- `values`: 物品价值列表 `[v0, v1, ...]`
- `capacity`: 背包容量上限

返回值规范：
- `status`: `"optimal"` 等
- `objective`: float 目标值（选入物品的总价值）
- `solution`: dict，键为物品下标（0..n-1），值为 0.0 或 1.0

In [ ]:
def solve(instance: dict) -> dict:
    weights = instance["weights"]
    values = instance["values"]
    capacity = instance["capacity"]
    n = instance["n"]

    model = Model("knapsack")
    model.hideOutput(True)

    # 0-1 变量：x_i 表示第 i 个物品是否装入背包
    x = [model.addVar(vtype="B", name=f"x_{i}") for i in range(n)]

    # 目标函数：最大化总价值
    model.setObjective(quicksum(values[i] * x[i] for i in range(n)), "maximize")

    # 容量约束
    model.addCons(quicksum(weights[i] * x[i] for i in range(n)) <= capacity)

    model.optimize()

    sol = {i: float(round(model.getVal(x[i]))) for i in range(n)}
    return {
        "status": "optimal" if model.getStatus() == "optimal" else str(model.getStatus()),
        "objective": float(model.getObjVal()),
        "solution": sol,
    }

In [ ]:
# 测试用例验证
test_instance = {
    "n": 4,
    "weights": [2, 3, 4, 5],
    "values": [3, 4, 5, 6],
    "capacity": 8
}
res = solve(test_instance)
print("求解结果:", res)